# **BankIntel AI Chatbot - BERT fine-tunning**

In [54]:
import pandas as pd
from transformers import AutoModel, AutoTokenizer

### **Data Loading**

In [2]:
train_df = pd.read_csv("dataset/train.csv")
test_df = pd.read_csv("dataset/test.csv")
val_df = pd.read_csv("dataset/dev.csv")

In [3]:
print(train_df.shape)
train_df.sample(5)

(10003, 3)


,text,label,label_text
716,Why does my transaction have an added fee?,34,extra_charge_on_statement
6037,How long is the wait for a money transfer to p...,48,pending_transfer
6055,How come I have pending transfers?,48,pending_transfer
9499,I used an ATM and there was an extra charge.,19,cash_withdrawal_charge
6621,How long will a transfer from the US take?,67,transfer_timing


In [4]:
print(test_df.shape)
test_df.sample(5)

(3080, 3)


,text,label,label_text
937,help me obtain a virtual card,40,getting_virtual_card
857,I need to find out why my transfer didn't get ...,66,transfer_not_received_by_recipient
2518,How do I get an actual card?,43,order_physical_card
2778,How much do I have to pay for the exchange fee?,31,exchange_charge
2286,Why was I not able to complete a transfer?,35,failed_transfer


In [5]:
print(val_df.shape)
val_df.sample(5)

(3080, 3)


,text,label,label_text
1557,I don't have my access code for the app.,44,passcode_forgotten
437,Can I use this app to change dollars into euros?,33,exchange_via_app
2099,I think my card payment had been return,53,reverted_card_payment?
812,im not sure what this charge is for,15,card_payment_fee_charged
2263,I received my salary in GBP. How do I change t...,50,receiving_money


### **Preprocessing**

In [6]:
train_df["label"].unique()

array([11, 13, 32, 17, 34, 46, 36, 12,  4, 14, 33, 41,  1, 49, 23, 56, 47,
        8, 60, 75, 15, 66, 54, 40, 10, 61,  6, 16, 30, 74, 68, 38, 73, 62,
       29, 22,  3, 28, 44, 26, 45, 42, 52, 27, 51, 25, 48, 55, 18, 63, 70,
       67, 53, 21,  7, 64, 50, 35, 65, 71, 39, 58, 43, 72, 76, 37, 59,  5,
       20, 31, 57,  0, 19,  9,  2, 69, 24])

In [7]:
label_df = pd.DataFrame({
    "label" : train_df['label'],
    "label_text" : train_df['label_text']
})

label_df_unique = label_df.drop_duplicates().sort_values("label")
label_map = dict(label_df_unique[['label', 'label_text']].values)

In [8]:
label_map

{0: 'activate_my_card',
 1: 'age_limit',
 2: 'apple_pay_or_google_pay',
 3: 'atm_support',
 4: 'automatic_top_up',
 5: 'balance_not_updated_after_bank_transfer',
 6: 'balance_not_updated_after_cheque_or_cash_deposit',
 7: 'beneficiary_not_allowed',
 8: 'cancel_transfer',
 9: 'card_about_to_expire',
 10: 'card_acceptance',
 11: 'card_arrival',
 12: 'card_delivery_estimate',
 13: 'card_linking',
 14: 'card_not_working',
 15: 'card_payment_fee_charged',
 16: 'card_payment_not_recognised',
 17: 'card_payment_wrong_exchange_rate',
 18: 'card_swallowed',
 19: 'cash_withdrawal_charge',
 20: 'cash_withdrawal_not_recognised',
 21: 'change_pin',
 22: 'compromised_card',
 23: 'contactless_not_working',
 24: 'country_support',
 25: 'declined_card_payment',
 26: 'declined_cash_withdrawal',
 27: 'declined_transfer',
 28: 'direct_debit_payment_not_recognised',
 29: 'disposable_card_limits',
 30: 'edit_personal_details',
 31: 'exchange_charge',
 32: 'exchange_rate',
 33: 'exchange_via_app',
 34: 'extr

In [9]:
tokenizer = AutoTokenizer.from_pretrained("./bert_model")

In [10]:
# Test with single sentence
tokenizer("Hello bert model", return_tensors='pt', padding='max_length')

{'input_ids': tensor([[  101,  7592, 14324,  2944,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [87]:
def text_encoder(df):
    encodings  = tokenizer(
        df['text'].tolist(), # text list expects
        truncation=True,
        padding=True,
    )

    return encodings

train_encodings = text_encoder(train_df)
test_encodings = text_encoder(test_df)
val_encodings = text_encoder(val_df)

In [98]:
train_labels = train_df['label'].tolist()
train_labels[1:10]

test_labels = test_df['label'].tolist()
val_labels = val_df['label'].tolist()

In [89]:
print(type(train_encodings))
# print(train_encodings.keys())
print(len(train_encodings['input_ids']))
print(len(train_encodings['input_ids'][0]))
# print(train_encodings.items())

<class 'transformers.tokenization_utils_base.BatchEncoding'>
10003
98


In [91]:
for k, v in train_encodings.items():
    # print(len(encodings['input_ids'][0]))
    print(k, len(v))

input_ids 10003
token_type_ids 10003
attention_mask 10003


In [92]:
print(len(train_encodings['input_ids'][0]))
# train_encodings['input_ids'][0]
# train_encodings['token_type_ids'][0]
# train_encodings['attention_mask'][0]

98


### **Creating Dataset**
- Creating a custom dataset by extending PyTorch’s built-in Dataset class

In [93]:
type(train_encodings.items())

collections.abc.ItemsView

In [94]:
list(train_encodings.keys())

['input_ids', 'token_type_ids', 'attention_mask']

In [95]:
train_encodings["input_ids"][0]

[101,
 1045,
 2572,
 2145,
 3403,
 2006,
 2026,
 4003,
 1029,
 102,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [96]:
import torch

# converts: Text + labels → PyTorch format
class BankingDataset(torch.utils.data.Dataset):   # inheriting (extending) PyTorch’s Dataset class
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    # When training starts, PyTorch calls __getitem__ , Eg : train_ds[0]
    def __getitem__(self, idx):
        item = {key : torch.tensor(val[idx]) for key, val in self.encodings.items()} # take one sample, convert to tensors, create dictionary
        item["labels"] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

In [103]:
# Creating obj of BankingDataset(in type pytorch Dataset), No tensor creation yet
train_ds = BankingDataset(train_encodings, train_labels)
test_ds = BankingDataset(test_encodings, test_labels) # final evaluation only
val_ds = BankingDataset(val_encodings, val_labels) # validation during training

In [101]:
type(train_ds)

__main__.BankingDataset

In [102]:
# For Hugging Face Trainer we must use 'labels' not 'label'
# if use pure PyTorch training loop since we can hanfle spelings does't mattter

train_ds[0]

{'input_ids': tensor([ 101, 1045, 2572, 2145, 3403, 2006, 2026, 4003, 1029,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0